# 1. Extraction and Preprocessing


Creating a miniature train file for developing and testing code faster:

In [1]:

def slice_conll(input_file, output_file, num_sentences=100):
    with open(input_file, "r",encoding='utf8') as file:
        content = file.read()
        
    #gets sentence units
    sentences = content.strip().split("\n\n")
    mini = sentences[:num_sentences]
    
    #writes them to new mini file
    with open(output_file, "w",encoding='utf8') as file:
        file.write("\n\n".join(mini) + "\n")
        
    
slice_conll("../data/en_ewt-up-train.conllu", "../data/mini_test.conllu")

Extracts the tokens with argument information in tuples, in lists which represent the sentences, in a final list where all the data is.

In [2]:

def extract_tokens_and_arg_info(conllfile):
    """    
    Reads a Conll file and extracts the sentences and tokens; where each sentence is made of tokens which are tupled with 
    argument information.
    Args:
        inputfile: a file in the Conllu format
    Returns:
        A list of sentences represented by a list of tokens, which are structured as tuples with their argument information."""
    
    allsentences = []
    currentsentence=[]
    with open(conllfile, 'r', encoding='utf8') as infile:
        for line in infile:
            #turn lines into strings
            line = line.strip()
            #sentence boundaries
            if line == '':
            #if boundary is reached and current sentence is full, append it to the larger list
                if currentsentence:
                    allsentences.append(currentsentence)
                    #restarts current sentence
                    currentsentence = []
            #if we are not at a boundary
            else:
                #if we are not at a metadata line
                if not line.startswith('#'):
                    #finds information per line
                    components = line.rstrip('\n').split()
                    index=components[0]
                    #if there are predicates in the sentence, since there would be more than 10 columns
                    #and excluding copies which have a dot in their index
                    if len(components)>10 and '.' not in index:
                        #for each columns of arguments, which starts from the tenth(indexing including id 
                        # and since index range is inclusive ) till the nth one
                        token=[components[1]]
                        for component in components[11:]:
                            #appends argument info to token
                            token.append(component)
                        #appends tuppled token information to sentence
                        currentsentence.append(tuple(token))
         
        
    return  allsentences 
            

Testing the extraction function


In [3]:
#test extraction
minisentences=extract_tokens_and_arg_info('../data/mini_test.conllu')
print(minisentences[:5])

[[('Al', '_'), ('-', '_'), ('Zaman', '_'), (':', '_'), ('American', '_'), ('forces', 'ARG0'), ('killed', 'V'), ('Shaikh', 'ARG1'), ('Abdullah', '_'), ('al', '_'), ('-', '_'), ('Ani', '_'), (',', '_'), ('the', '_'), ('preacher', '_'), ('at', '_'), ('the', '_'), ('mosque', 'ARGM-LOC'), ('in', '_'), ('the', '_'), ('town', '_'), ('of', '_'), ('Qaim', '_'), (',', '_'), ('near', '_'), ('the', '_'), ('Syrian', '_'), ('border', '_'), ('.', '_')], [('[', '_', '_', '_', '_'), ('This', '_', '_', '_', '_'), ('killing', 'V', '_', 'ARG0', '_'), ('of', '_', '_', '_', '_'), ('a', '_', '_', '_', '_'), ('respected', '_', '_', '_', '_'), ('cleric', 'ARG1', '_', '_', '_'), ('will', '_', '_', 'ARGM-MOD', '_'), ('be', '_', 'V', '_', '_'), ('causing', '_', '_', 'V', '_'), ('us', '_', '_', 'ARGM-GOL', '_'), ('trouble', '_', '_', 'ARG1', '_'), ('for', '_', '_', '_', '_'), ('years', '_', '_', 'ARGM-TMP', 'ARG1'), ('to', '_', '_', '_', '_'), ('come', '_', '_', '_', 'V'), ('.', '_', '_', '_', '_'), (']', '_', '_'

Preprocessing: A sentence may have more than one predicate. To solve this, replicate
each sentence as many times as there are predicates so each training instance has a
single labeled argument structure 


In [4]:
import csv

def data_preprocessing(extracteddata,outputfile):
    ''' Preprocesses the data extracted from the Conllu files, multiples sentences with multiple predicates in the dataset,
    finds the verb, and the gold labels per predicate.
    Args:
        inputfile: output of the function extract_tokens_and_arg_info(); or data in the following format: list of lists(sentences) 
        of tuples(tokens) containing the token and all its labels of arguments in that sentence. [[(token, arginfo,+)]]
        
    Returns: list of lists-sentences with respectives repetitions according to number of verbs,
        list of lists- goldaarguments lists, one per sentence/repeated sentence
        list- the token of the predicate corresponding to the list of gold labels'''
    
    verbs=[]
    tokenonlysentences=[]
    goldarguments=[]
    
    for sentence in extracteddata:
        #to further avoid cases with no verbs, checks for the lenght
        if len(sentence[0])<2:
            continue
        else:
            tokensentence=[]
            #finds number of verbs through lenght of tuple per sentence
            numofverbs=len(sentence[0])-1
            
            if len(sentence)>1:                
                
            #starts sorting the argument information of each token into separate tuples    
                sortedargumentinfo = [list(pair) for pair in zip(sentence[0][1:], sentence[1][1:])]
                #if there are more than two tokens
                if len(sentence)>2:
                    #adds info per token by ennumerating the elements of each token and appending them to the lists of the respective index
                    for tokentuple in sentence[2:]:
                        for i, addition in enumerate(tokentuple[1:]):
                            sortedargumentinfo[i].append(addition)
                
                
                
            #appends each sorted argument list into the list of gold labels per sentence/duplicate sentence  
                for argumentlist in sortedargumentinfo:
                    goldarguments.append(argumentlist)
                    

                for tokentuple in sentence:
                    token=tokentuple[0]
                    tokensentence.append(token)
                    

                #after tokens have been appended to each sentence, appends the full sentence as many times as there are verbs
                for i in range(numofverbs):
                    tokenonlysentences.append(tokensentence)
                    
    #finds the index of the verb in a sentence, gets it, appends it to list of verbs, in the correct order
    for sentence, arguments in zip(tokenonlysentences,goldarguments):
        if 'V' in arguments:
            indexofverb=arguments.index('V')
            verb=sentence[indexofverb]
            verbs.append(verb)
        
        
  
    for goldlabellist in goldarguments:
        #remove V from gold labels, with indexing
        goldlabellist[goldlabellist.index("V")] = "_"

    
    #writing to a file
    rows=[]
    #appends first the tokens and labels, and keeps the index for later appending the verb
    for groupindex, (sentences, goldargs) in enumerate(zip(tokenonlysentences, goldarguments)):
        for token, goldlabel in zip(sentences, goldargs):
            rows.append((groupindex,token, goldlabel))

    with open(outputfile, "w", encoding='utf8',newline="" ) as file:
        writer = csv.writer(file, delimiter="\t")
        for groupindex, token, goldlabel in rows:
            writer.writerow([token, goldlabel])
    
    with open(outputfile, encoding='utf8',newline="") as file:
        tokenlabel = list(csv.reader(file, delimiter="\t"))

    with open(outputfile, "w", encoding='utf8',newline="") as file:
        writer = csv.writer(file, delimiter="\t")
        for (group_index,token, goldlabel), row in zip(rows, tokenlabel):
            #uses the index found in the rows list for appending the correct verb to each token row
            writer.writerow(row + [verbs[group_index]])
 
    return verbs


    

Testing the function with the mini set


In [5]:


def readpreprocessedfile(preprocessedfile):
    
    tokenonlysentences, goldarguments,verbs = [], [], []

    with open(preprocessedfile, newline="", encoding='utf8') as file:
        reader = csv.reader(file, delimiter="\t")
        for row in reader:
            tokenonlysentences.append(row[0])
            goldarguments.append(row[1])
            verbs.append(row[2])
 
    return tokenonlysentences, goldarguments,verbs





verbs_mini=data_preprocessing(minisentences,'../data/mini_test_preprocessed.tsv')

tokenonlysentences_mini, goldarguments_mini,repeated_verbs_mini=readpreprocessedfile('../data/mini_test_preprocessed.tsv')


print(len(tokenonlysentences_mini),len(goldarguments_mini),len(verbs_mini), len(repeated_verbs_mini))
print(goldarguments_mini)


14948 14948 435 14948
['_', '_', '_', '_', '_', 'ARG0', '_', 'ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', 'ARGM-LOC', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', 'ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', 'ARG0', '_', '_', '_', '_', 'ARGM-MOD', '_', '_', 'ARGM-GOL', 'ARG1', '_', 'ARGM-TMP', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', 'ARG1', '_', '_', '_', '_', '_', '_', '_', 'ARG0', '_', '_', '_', '_', 'ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', 'ARG0', '_', '_', '_', '_', '_', 'ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', 'ARG0', '_', '_', 'ARGM-LOC', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_',

Running the extraction and processing on the training set + small tests



In [6]:

train_sents=extract_tokens_and_arg_info('../data/en_ewt-up-train.conllu')
print(train_sents[:5])

verbs_train=data_preprocessing(train_sents,'../data/train_preprocessed.tsv')

#verbs are repeated per token per sentence
tokens_preprocessed_train, goldarguments_train, repeated_verbs_train=readpreprocessedfile('../data/train_preprocessed.tsv')

print(len(tokens_preprocessed_train),len(repeated_verbs_train), len(goldarguments_train))
print(goldarguments_train[:30])
print(len(train_sents))
print(len(verbs_train))

[[('Al', '_'), ('-', '_'), ('Zaman', '_'), (':', '_'), ('American', '_'), ('forces', 'ARG0'), ('killed', 'V'), ('Shaikh', 'ARG1'), ('Abdullah', '_'), ('al', '_'), ('-', '_'), ('Ani', '_'), (',', '_'), ('the', '_'), ('preacher', '_'), ('at', '_'), ('the', '_'), ('mosque', 'ARGM-LOC'), ('in', '_'), ('the', '_'), ('town', '_'), ('of', '_'), ('Qaim', '_'), (',', '_'), ('near', '_'), ('the', '_'), ('Syrian', '_'), ('border', '_'), ('.', '_')], [('[', '_', '_', '_', '_'), ('This', '_', '_', '_', '_'), ('killing', 'V', '_', 'ARG0', '_'), ('of', '_', '_', '_', '_'), ('a', '_', '_', '_', '_'), ('respected', '_', '_', '_', '_'), ('cleric', 'ARG1', '_', '_', '_'), ('will', '_', '_', 'ARGM-MOD', '_'), ('be', '_', 'V', '_', '_'), ('causing', '_', '_', 'V', '_'), ('us', '_', '_', 'ARGM-GOL', '_'), ('trouble', '_', '_', 'ARG1', '_'), ('for', '_', '_', '_', '_'), ('years', '_', '_', 'ARGM-TMP', 'ARG1'), ('to', '_', '_', '_', '_'), ('come', '_', '_', '_', 'V'), ('.', '_', '_', '_', '_'), (']', '_', '_'

Provide a set of statistics over the train and test set including, for each dataset, the number
of tokens and number of sentences before and after preprocessing the datasets to
deal with sentences containing multiple predicates.


In [7]:
#TRAIN

num_sent_raw_train=len(train_sents)

tokencount_raw_train= 0
for sentence in train_sents:
    for token in sentence:
        tokencount_raw_train += 1
        
        
num_sent_preprocessed_train=len(verbs_train)



#DEV
dev_sents=extract_tokens_and_arg_info('../data/en_ewt-up-dev.conllu')

#creates preprocessing file and also returns the list of unrepeated verbs
verbs_dev=data_preprocessing(dev_sents,'../data/dev_preprocessed.tsv')

#verbs are repeated per token per sentence
tokens_preprocessed_dev, goldarguments_dev,repeated_verbs_dev=readpreprocessedfile('../data/dev_preprocessed.tsv')


num_sent_raw_dev=len(dev_sents)

tokencount_raw_dev= 0
for sentence in dev_sents:
    for token in sentence:
        tokencount_raw_dev += 1
        
        
num_sent_preprocessed_dev=len(verbs_dev)



#TEST  
   
test_sents=extract_tokens_and_arg_info('../data/en_ewt-up-test.conllu')

#creates preprocessing file and also returns the list of unrepeated verbs
verbs_test=data_preprocessing(test_sents,'../data/test_preprocessed.tsv')

#verbs are repeated per token per sentence
tokens_preprocessed_test, goldarguments_test,repeated_verbs_test=readpreprocessedfile('../data/test_preprocessed.tsv')


num_sent_raw_test=len(test_sents)

tokencount_raw_test= 0
for sentence in test_sents:
    for token in sentence:
        tokencount_raw_test += 1
        
        
num_sent_preprocessed_test=len(verbs_test)

#PRINT STATEMENTS


print('The following table shows the number of raw vs preprocessed sentences and tokens per dataset (train, test, dev)\n-------------------------------------------------------------------------------')


print(f"{'Dataset':<15} {'Raw sents':<12} {'Processed sents':<16} {'Raw tokens':<12} {'Processed tokens':<16}")
print(f"{'train':<15} {num_sent_raw_train:<12} {num_sent_preprocessed_train:<16} {tokencount_raw_train:<12} {len(tokens_preprocessed_train):<16}")
print(f"{'dev':<15} {num_sent_raw_dev:<12} {num_sent_preprocessed_dev:<16} {tokencount_raw_dev:<12} {len(tokens_preprocessed_dev):<16}")
print(f"{'test':<15} {num_sent_raw_test:<12} {num_sent_preprocessed_test:<16} {tokencount_raw_test:<12} {len(tokens_preprocessed_test):<16}")

The following table shows the number of raw vs preprocessed sentences and tokens per dataset (train, test, dev)
-------------------------------------------------------------------------------
Dataset         Raw sents    Processed sents  Raw tokens   Processed tokens
train           12542        40451            204550       1028106         
dev             1974         4968             24969        105059          
test            2062         4781             25009        101126          


# 2. Feature Engineering

You must motivate and extract EXACTLY three features (not more and not less). There
is no such thing as “base/core/default” features.
- One of the features you need to motivate and extract is given (same for everyone):
This is a complex feature integrating: the directed dependency path from the token to the
predicate + the predicate’s lemma;



In [8]:


import spacy
from spacy.tokens import Doc

nlp=spacy.load("en_core_web_sm")

def feature_extraction_from_preprocessed_data(tokens_nosentence,repeatedverbs):
    """Extracts three features off of each token of the preprocessed data, into feature dictionaries. 
    Features: Lemma, entity label, and path to root + verb lemma.
    Args:
        tokens_nosentence: list of tokens without sentence boundaries, extracted from preprocessed doc
        repeatedverbs: list of verbs without sentence boundaries, extracted from preprocessed doc, These lists shoudl be the same size
        
    Returns: List of feature dictionaries per token, not keeping sentence structure.
    """
    
    
    listoffeatdicts=[]
    
    #to find current sentence boundaries
    
    sentences = []
    current_sentence = []
    verbs_tozip = []
    current_verb = repeatedverbs[0]

    for token, verb in zip(tokens_nosentence, repeatedverbs):
        if verb == current_verb:
            current_sentence.append(token)
        else:
            verbs_tozip.append(current_verb)
            sentences.append(current_sentence)
            current_sentence = [token]
            current_verb = verb

    sentences.append(current_sentence)
    verbs_tozip.append(current_verb)
    
    
    #per sentence in the dataset, creates a doc
    for sentence,targetverb in zip(sentences,verbs_tozip):
        #to bypass the tokenizer, and feed it the already tokenized sentences
        doc = Doc(nlp.vocab, words=sentence)
        for name, component in nlp.pipeline:
            doc = component(doc)
                
        #find position of verb token, to use it as a spacy element    
        verb_token = None
        for item in doc:
            if item.text == targetverb:
                verb_token = item
                break
                  
        for token in doc:
            lemma=token.lemma_ 
            #when its outside it returns an empty string, to make it return something other than false,
            if token.ent_type_:
                nertag=token.ent_type_
            else:
                nertag='O'
            
            #creating feature dict per token
            feature_dictionary={'lemma': lemma,
                                'nertag': nertag}
            
            
        
    
            """ Complex Path Feature -----------------------------------------------------------------------------------"""
            
            #for each token, we start with itself, and append its index, its shape, and its deprel
            pathfromtoken =[(token.i, token.text) ]
            #a separate list for the path only with dependeny relations
            pathfromtokendeprels=[token.dep_]
            #for each ancestor, which goes up to the root, we do the same
            for ancestor in token.ancestors:
                pathfromtoken.append((ancestor.i, ancestor.text))
                pathfromtokendeprels.append(ancestor.dep_)
            
            
            #we do the same process for the verb associated with the current token
            pathfromverb =[(verb_token.i, verb_token.text)] 
            #a separate list for the path only with dependeny relations
            pathfromverbdeprels=[verb_token.dep_]
            for ancestor in verb_token.ancestors:
                pathfromverb.append((ancestor.i, ancestor.text))
                pathfromverbdeprels.append(ancestor.dep_)
                    
          
            #now we find the common ancestor in the paths, by comparing both the text and the index 
            #to ensure that the head is the same and not only the same shape
            for index, infotupletoken in enumerate(pathfromtoken):
                #find the index of the common head IN THE PATH lists (not in the regular sentence)
                if infotupletoken in pathfromverb:
                    boundaryindex_verbpath=pathfromverb.index(infotupletoken)
                    #goes untill the boundary index in the dependency relation path list
                    path_up = pathfromtokendeprels[:index+1]
                    
                    #does the opposite from the verb, by inverting that deprel list
                    path_down = pathfromverbdeprels[:boundaryindex_verbpath][::-1]
                    
                    #adding arrows
                    path_up_arrows="↑".join(path_up)
                    path_down_arrows="↓".join(path_down)
                    #joining both paths and verb lemma
                    completepathstring=f"{path_up_arrows}↓{path_down_arrows}↓{verb_token.lemma_}"
                    #adds to dictionary
                    feature_dictionary['pathtoverb']=completepathstring
                
            #appends dict per token
            listoffeatdicts.append(feature_dictionary)

    return listoffeatdicts






Dependency path from token to predicate:

The path from token to predicate in a dependency structure is informative for SRL as this positioning information points to the relationship of the token with the predicate. The position and relation of the arguments to a predicate show patterns which help the argument classifcation as well as the later choice for semantic role. 
For this complex feature, I used the following steps:

1- get the current token and corresponding verb, and make them into spacy elements.
2- get two lists of the ancestors of the token and of the verb with the spacy method .ancestors, which goes from the current token to the root. The first of which has the the token and index in the sentence, and the second which is only the dependency relationships.
3- find the common ancestor of each token, by seeing if the tuple of token and index is in the list of ancestors of the verb. Index is also used, since ther might be repeated tokens in the sentence.
4- find the position of the ancestor in the list of ancestors of both the token and verb. one way it will be going up, and another it will go down.
5- join the paths by using the boundary index, inverting the path to the verb, and adding the direction arrows when joining. It also adds the lemma of the current verb.

A complete feature should look something like:

for Kill, and lemma '-'
{'pathtoverb': 'punct↑ROOT↓acl↓kill'}


Lemma of the current token:

Using the shape of the current token is informative for SRL since specific words may have a characteristic argument behaviour, which might not be clear only from its dependency path to the predicate. For example, in ' the mother is cooking' and 'the potatos are cooking', the noun in the beginning of the sentece will imply different arguments, even though they share a lot of the same characteristics. To go one step further and hopefully make the feature more general, I chose to use the lemma instead of the token shape itself. This way, behaviours of plurals and inflections of the same stem can hopefully be analysed together. It is extracted from the spacy processing of the sentences, by accessing the lemma information deduced by their lemmatizer. 
This feature is extracted as strings of the respective lemma.

Zaman: {'lemma': 'Zaman'}

Killed: {'lemma': 'kill'}


NER label of the current token:

The NER label of the current token is informative for SRL, specifically for choosing between argument types. Recalling the example used in the previour feature, including the information that a token is an entity (person) or not (outside) could help the model know that since 'mother' is a person, they could be ARG0/ the agent, and the 'potatos' could not. Just as the previous feature, this one is extracted directly from the token with spacy. 
This feature is extracted and can have either an entity tag or an 'O' tag when it is not recognized as an entity.

Zaman: {'nertag': 'PERSON'}

Killed: {'nertag': 'O'}

Printing first three sentences' feature dictionaries. Uses indexing since list of features does not keep sentence structure. Ran on the mini sample.

In [9]:

examplefeatureextraction=feature_extraction_from_preprocessed_data(tokenonlysentences_mini,repeated_verbs_mini)

for featdict in examplefeatureextraction[:64]:
    print(featdict)
    


{'lemma': 'Al', 'nertag': 'PERSON', 'pathtoverb': 'compound↑ROOT↓acl↓kill'}
{'lemma': '-', 'nertag': 'PERSON', 'pathtoverb': 'punct↑ROOT↓acl↓kill'}
{'lemma': 'Zaman', 'nertag': 'PERSON', 'pathtoverb': 'ROOT↓acl↓kill'}
{'lemma': ':', 'nertag': 'O', 'pathtoverb': 'punct↑ROOT↓acl↓kill'}
{'lemma': 'american', 'nertag': 'NORP', 'pathtoverb': 'amod↑nsubj↑acl↑ROOT↓acl↓kill'}
{'lemma': 'force', 'nertag': 'O', 'pathtoverb': 'nsubj↑acl↑ROOT↓acl↓kill'}
{'lemma': 'kill', 'nertag': 'O', 'pathtoverb': 'acl↑ROOT↓acl↓kill'}
{'lemma': 'Shaikh', 'nertag': 'PERSON', 'pathtoverb': 'compound↑dobj↑acl↑ROOT↓acl↓kill'}
{'lemma': 'Abdullah', 'nertag': 'PERSON', 'pathtoverb': 'compound↑dobj↑acl↑ROOT↓acl↓kill'}
{'lemma': 'al', 'nertag': 'PERSON', 'pathtoverb': 'compound↑dobj↑acl↑ROOT↓acl↓kill'}
{'lemma': '-', 'nertag': 'PERSON', 'pathtoverb': 'punct↑dobj↑acl↑ROOT↓acl↓kill'}
{'lemma': 'Ani', 'nertag': 'PERSON', 'pathtoverb': 'dobj↑acl↑ROOT↓acl↓kill'}
{'lemma': ',', 'nertag': 'O', 'pathtoverb': 'punct↑dobj↑acl↑ROO

# Model creation and evaluation
(NB. This section contains some portions of code inspired or taken from the machine learning notebooks)

Importing modules for model creation and evaluation

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import classification_report
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


Run one classification experiment using Sklearn’s LogisticRegression by training a
token-level SRL on the Universal Propositions Bank v1.0 English dataset.


In [ ]:

def create_classifier(train_features, train_targets):
    """
    Creates a classifier using dictvectorize by converting the training features dictionary and targets into vectors and training a model with them.
    Args:
        train_features: list of dictionaries per token with the features for training
        train_targets: golden labels
    Returns:
        Model and respective vectorizer"""
    #Increased iterations to allow model to converge
    logreg = LogisticRegression(max_iter=1000,     
            # saga is used since i kept getting a memory error due to the large and sparse data. found online as a solution    
            solver='saga')
    vec = DictVectorizer()
    features_vectorized = vec.fit_transform(train_features)
    model = logreg.fit(features_vectorized, train_targets)
    
    return model, vec

In [12]:
train_features=feature_extraction_from_preprocessed_data(tokens_preprocessed_train,repeated_verbs_train)



In [13]:
#checking if dimentions line up
print(len(train_features))
print(len(tokens_preprocessed_train))
print(len(repeated_verbs_train))
print(len(goldarguments_train))

1028106
1028106
1028106
1028106


In [14]:


#creating classifier with training data

logreg_classifier_a1, vec_logreg_a1=create_classifier(train_features,goldarguments_train)


In [ ]:
import pickle

#saves both objects together
with open("../models/logreg_model_a1.pkl","wb") as file:
    pickle.dump({
        "classifier": logreg_classifier_a1,
        "vectorizer": vec_logreg_a1
    }, file)

Evaluate the model you produce on the test set. You must use Scikit-learn’s
Classification Report to provide Precision, Recall and F1 measures for token-level
classification. Make sure to also include a labeled confusion matrix that supports the
classification metrics.
Store your model’s predictions over the preprocessed testset and save it in a
human-readable text format (e.g., as a tsv). Make sure this file contains, at least, 
token, the gold label and the predicted label for each prediction.


In [16]:
def classify_data(model, vec, inputdata, outputfile):
    """ Uses previous functions to extract the features and vectorize test data. Tests the model on the data, and outputs the results to a tsv file.
    Args:
        model: model we wish to test
        vec: vectorizer, same as for the training
        inputdata: test data, with extracted features
        outputfile: location to create file with output
    Returns:
        File with predicted results"""
    

    #extract features from dev data
    tokens,gold,verbs=readpreprocessedfile(inputdata)
    dataextractedfeatures = feature_extraction_from_preprocessed_data(tokens,verbs)
    featuresvec = vec.transform(dataextractedfeatures)
    predictions = model.predict(featuresvec)
    outfile = open(outputfile, 'w',encoding='utf-8')
    counter = 0
    for line in open(inputdata, 'r',encoding='utf-8'):
        if len(line.rstrip('\n').split()) > 0:
            outfile.write(line.rstrip('\n') + '\t' + predictions[counter] + '\n')
            counter += 1
    outfile.close()

In [ ]:
#get the prepreocessed data from the dev data and then extract features of it
classify_data(logreg_classifier_a1, vec_logreg_a1, "../data/dev_preprocessed.tsv","../models/logreg_predictions")

In [ ]:
#repeat for test data
classify_data(logreg_classifier_a1, vec_logreg_a1, "../data/test_preprocessed.tsv","../models/logreg_testdata_predictions")

Gets the data of the predictions file that was just created, for classification reports


In [18]:
def extract_classification_data(modelpredictionfile):
    """    
    Reads a Conll file with prediction labels and extracts the gold and predicted label
    """
    data = []
    gold = []
    predictions=[]
    
    with open(modelpredictionfile, 'r', encoding='utf8') as infile:
        for line in infile:
            components = line.rstrip('\n').split()
            if len(components) > 0:
                token = components[0]
                data.append(token)
                gold.append(components[-3])
                predictions.append(components[-1])
    return data, gold, predictions

In [ ]:

data_logreg, gold_logreg, predictions_logreg=extract_classification_data('../models/logreg_predictions')
print(gold_logreg[:10])
print(predictions_logreg[:10])
print(len(gold_logreg), len(predictions_logreg))

['_', '_', 'ARG2', '_', '_', 'ARG1', '_', 'ARG0', '_', '_']
['_', '_', '_', '_', '_', '_', '_', '_', '_', '_']
105059 105059


In [ ]:
data_logreg, gold_logreg, predictions_logreg=extract_classification_data('../models/logreg_predictions')

report = classification_report(gold_logreg,predictions_logreg,digits = 3)
print(' classifier labels report----------------------------------------------------------------')
print(report)

cf_matrix_logreg_data = confusion_matrix(gold_logreg,predictions_logreg)
print('')
print('')
print(cf_matrix_logreg_data)
print('')
print('')
display = ConfusionMatrixDisplay(confusion_matrix=cf_matrix_logreg_data)
#display.plot()

c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this b

 classifier labels report----------------------------------------------------------------
              precision    recall  f1-score   support

        ARG0      0.828     0.340     0.482      1733
        ARG1      0.833     0.361     0.503      3325
        ARG2      0.765     0.399     0.524      1212
        ARG3      0.500     0.026     0.049        77
        ARG4      0.360     0.191     0.250        47
        ARG5      0.000     0.000     0.000         1
    ARGM-ADJ      0.923     0.096     0.173       251
    ARGM-ADV      0.629     0.137     0.225       481
    ARGM-CAU      0.667     0.057     0.105        70
    ARGM-COM      0.000     0.000     0.000        14
    ARGM-CXN      0.000     0.000     0.000        14
    ARGM-DIR      0.500     0.021     0.040        48
    ARGM-DIS      0.896     0.231     0.368       186
    ARGM-EXT      0.864     0.168     0.281       113
    ARGM-GOL      0.000     0.000     0.000        26
    ARGM-LOC      0.583     0.029     0.055  

In [ ]:
data_logreg, gold_logreg, predictions_logreg=extract_classification_data('../models/logreg_testdata_predictions')

report = classification_report(gold_logreg,predictions_logreg,digits = 3)
print(' classifier labels report test data----------------------------------------------------------------')
print(report)

cf_matrix_logreg_data = confusion_matrix(gold_logreg,predictions_logreg)
print('')
print('')
print(cf_matrix_logreg_data)
print('')
print('')
display = ConfusionMatrixDisplay(confusion_matrix=cf_matrix_logreg_data)
#display.plot()

c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this b

 classifier labels report test data----------------------------------------------------------------
              precision    recall  f1-score   support

        ARG0      0.853     0.391     0.536      1733
        ARG1      0.834     0.370     0.513      3241
    ARG1-DSP      0.000     0.000     0.000         4
        ARG2      0.754     0.407     0.529      1129
        ARG3      1.000     0.095     0.173        74
        ARG4      0.577     0.268     0.366        56
        ARG5      0.000     0.000     0.000         1
        ARGA      0.000     0.000     0.000         2
    ARGM-ADJ      0.857     0.105     0.188       228
    ARGM-ADV      0.620     0.161     0.256       496
    ARGM-CAU      0.800     0.087     0.157        46
    ARGM-COM      0.000     0.000     0.000        13
    ARGM-CXN      0.000     0.000     0.000        12
    ARGM-DIR      0.000     0.000     0.000        47
    ARGM-DIS      0.833     0.247     0.381       182
    ARGM-EXT      0.870     0.190  

loading the model and vec

In [ ]:
with open("../models/logreg_model_a1.pkl","rb") as file:
    bundle=pickle.load(file)

logreg_classifier_a1=bundle["classifier"]
vec_logreg_a1=bundle["vectorizer"]

Final ready to use function

In [ ]:
def model_testing_through_example(sentence,predicate_position,model=logreg_classifier_a1, vectorizer=vec_logreg_a1):
    """Allows a user to provide a single sentence and verb position in said sentence, as well as a list of sentences
    and a corresponding list of verbs, and get the model prediction for it
        Args:
        sentence: list of tokens 
        predicate position: a list of integers, with a SINGLE 1 mdenoting the position in the sentence of the target verb
        model:defaults to logreg_classifier_a1
        vectorizer: defaults to vec_logreg_a1
        """
     
    verbindex=predicate_position.index(1)
    verb=sentence[verbindex]
    #to format it in a way that the function can unpack...
    verbintolist=[verb]*len(sentence)
    
    extractedfeatures_currentsentence=feature_extraction_from_preprocessed_data(sentence,verbintolist)
    
    sentenceintovec = vec_logreg_a1.transform(extractedfeatures_currentsentence)
    
    prediction = logreg_classifier_a1.predict(sentenceintovec)   


    return prediction
        

Example use of the function:

NB: the position list should be with integers. Both lists need to be the same size. The output should be printed, as the function only returns the predictions.

In [31]:
print(model_testing_through_example(['Pia', 'asked', 'Luis', 'to', 'write', 'this', 'sentence','.'],[0,1,0,0,0,0,0,0]))

['ARG0' '_' 'ARG2' '_' 'ARG1' '_' '_' '_']


Submit one zip file containing a Jupyter notebook (and HTML printout) accompanied
by a requirements.txt file and any number of python modules with helper functions. Please
make sure to start from a clean environment and only use external modules you need.
Include also your model’s predictions on the testset (e.g., as a tsv). Do not upload the saved
model on Canvas! Provide a link to download the model instead (e.g. needs to be a
public link) – make sure the link is available at the top of your notebook. Make sure you run
the notebook and save the notebook with the output of all cells. Read more information
about the requirements below.
